本程式使用pyautogui來完成一個blitz的遊戲機器人

In [10]:
import pyautogui as pyg
pyg.sleep(2)
basex=428
basey=560
gap=40

In [13]:
def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的 (x, y) 座標，索引範圍為 0 到 7。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError("r 和 c 必須介於 0 到 7")
    return (basex + c * gap, basey + r * gap)

In [5]:
def gem_position(r,c):
    x=basex+c*gap
    y=basey+r*gap
    return(x,y)


In [6]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))

(298, 442)
(338, 442)
(298, 482)


In [14]:
for i in range(8):
    x,y=gem_position(0,i)
    pyg.click(x,y,duration=1)

In [11]:
def get_gem_color(r,c,im):
    position=gem_position(r,c)
    return im.getpixel(position)

In [8]:
def can_move(r, c, im):
    """判斷第 r 列、第 c 欄的寶石往左交換後是否能形成三連。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError("r 和 c 必須介於 0 到 7")
    if c == 0:
        return False

    colors = [[get_gem_color(row, col, im) for col in range(8)]
              for row in range(8)]
    if colors[r][c] == colors[r][c - 1]:
        return False

    # 模擬和左側寶石交換，不實際操作滑鼠。
    colors[r][c], colors[r][c - 1] = colors[r][c - 1], colors[r][c]

    def has_match_at(row, col):
        color = colors[row][col]
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and colors[nr][nc] == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    # 交換後新形成的三連必定經過其中一個交換位置。
    return has_match_at(r, c) or has_match_at(r, c - 1)

In [15]:
im=pyg.screenshot()
print(can_move_left(2,2,im))

NameError: name 'can_move_left' is not defined

In [17]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [18]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            directions = (
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2)
                    pyg.sleep(0.2)
                    im = pyg.screenshot()
                    break

        if time.time() - start_time > 60:
            break

NameError: name 'can_move_left' is not defined